In [14]:
import pandas as pd

# Load the file specifying latin1 encoding
df = pd.read_csv('Sample - Superstore.csv', encoding='latin1')

# View the dataset
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [2]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
# Check dataset dimensions and missing values
print("Dataset Shape:", df.shape)
print("\nMissing Values Count:")
print(df.isnull().sum())

Dataset Shape: (9994, 21)

Missing Values Count:
Row ID           0
Order ID         0
Order Date       0
Ship Date        0
Ship Mode        0
Customer ID      0
Customer Name    0
Segment          0
Country          0
City             0
State            0
Postal Code      0
Region           0
Product ID       0
Category         0
Sub-Category     0
Product Name     0
Sales            0
Quantity         0
Discount         0
Profit           0
dtype: int64


In [4]:
# Convert text dates to proper datetime objects
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date'] = pd.to_datetime(df['Ship Date'])

# View datatypes to confirm conversion
df.dtypes[['Order Date', 'Ship Date', 'Sales', 'Profit']]

Order Date    datetime64[ns]
Ship Date     datetime64[ns]
Sales                float64
Profit               float64
dtype: object

In [5]:
# Group data by Product Name
product_summary = df.groupby('Product Name').agg(
    Category=('Category', 'first'),
    Sub_Category=('Sub-Category', 'first'),
    Total_Sales=('Sales', 'sum'),
    Total_Quantity=('Quantity', 'sum'),
    Total_Profit=('Profit', 'sum'),
    Total_Orders=('Order ID', 'count')
).reset_index()

# Sort products from highest to lowest revenue
product_summary = product_summary.sort_values(by='Total_Sales', ascending=False)

# Categorize products into 3 equal tiers using quantiles
product_summary['Performance_Tier'] = pd.qcut(
    product_summary['Total_Sales'],
    q=3,
    labels=['Low Performer', 'Mid Performer', 'Top Performer']
)

# View the top 5 product performers
product_summary.head()

,Product Name,Category,Sub_Category,Total_Sales,Total_Quantity,Total_Profit,Total_Orders,Performance_Tier
404,Canon imageCLASS 2200 Advanced Copier,Technology,Copiers,61599.824,20,2.519993e+04,5,Top Performer
650,Fellowes PB500 Electric Punch Plastic Comb Bin...,Office Supplies,Binders,27453.384,31,7.753039e+03,10,Top Performer
444,Cisco TelePresence System EX90 Videoconferenci...,Technology,Machines,22638.480,6,-1.811078e+03,1,Top Performer
786,HON 5400 Series Task Chairs for Big and Tall,Furniture,Chairs,21870.576,39,5.684342e-14,8,Top Performer
686,GBC DocuBind TL300 Electric Binding System,Office Supplies,Binders,19823.479,37,2.233505e+03,11,Top Performer


In [6]:
# Print breakdown of counts per tier
print("Product Distribution Across Tiers:")
print(product_summary['Performance_Tier'].value_counts())

# Save cleaned output file for SQL loading
product_summary.to_csv('product_performance_summary.csv', index=False)
print("\nFile 'product_performance_summary.csv' saved successfully!")

Product Distribution Across Tiers:
Performance_Tier
Low Performer    617
Top Performer    617
Mid Performer    616
Name: count, dtype: int64

File 'product_performance_summary.csv' saved successfully!


In [7]:
import sqlite3
import pandas as pd

# Connects automatically — no password required!
conn = sqlite3.connect('superstore_analytics.db')

# Load raw and summary CSVs
df_raw = pd.read_csv('Sample - Superstore.csv', encoding='latin1')
df_summary = pd.read_csv('product_performance_summary.csv')

# Load tables into SQL
df_raw.to_sql('orders_raw', conn, if_exists='replace', index=False)
df_summary.to_sql('product_performance', conn, if_exists='replace', index=False)

print("Database and tables created successfully with SQLite!")

Database and tables created successfully with SQLite!


In [8]:
# Helper function to run SQL queries and display output as a DataFrame
def run_sql(query):
    return pd.read_sql_query(query, conn)

# Verify table names inside the database
run_sql("SELECT name FROM sqlite_master WHERE type='table';")

,name
0,orders_raw
1,product_performance


In [9]:
query_1 = """
SELECT 
    Performance_Tier,
    COUNT("Product Name") AS Total_Products,
    ROUND(SUM(Total_Sales), 2) AS Tier_Revenue,
    ROUND(SUM(Total_Profit), 2) AS Tier_Profit,
    ROUND((SUM(Total_Profit) / SUM(Total_Sales)) * 100, 2) AS Profit_Margin_Pct
FROM product_performance
GROUP BY Performance_Tier
ORDER BY Tier_Revenue DESC;
"""

run_sql(query_1)

,Performance_Tier,Total_Products,Tier_Revenue,Tier_Profit,Profit_Margin_Pct
0,Top Performer,617,2037585.04,231595.32,11.37
1,Mid Performer,616,215728.05,42971.21,19.92
2,Low Performer,617,43887.78,11830.49,26.96


In [10]:
query_2 = """
SELECT 
    "Product Name",
    Category,
    ROUND(Total_Sales, 2) AS Revenue,
    Total_Quantity,
    ROUND(Total_Profit, 2) AS Profit,
    Performance_Tier
FROM product_performance
ORDER BY Total_Sales DESC
LIMIT 5;
"""

run_sql(query_2)

,Product Name,Category,Revenue,Total_Quantity,Profit,Performance_Tier
0,Canon imageCLASS 2200 Advanced Copier,Technology,61599.82,20,25199.93,Top Performer
1,Fellowes PB500 Electric Punch Plastic Comb Bin...,Office Supplies,27453.38,31,7753.04,Top Performer
2,Cisco TelePresence System EX90 Videoconferenci...,Technology,22638.48,6,-1811.08,Top Performer
3,HON 5400 Series Task Chairs for Big and Tall,Furniture,21870.58,39,0.00,Top Performer
4,GBC DocuBind TL300 Electric Binding System,Office Supplies,19823.48,37,2233.51,Top Performer


In [11]:
query_3 = """
SELECT 
    "Product Name",
    Category,
    ROUND(Total_Sales, 2) AS Revenue,
    Total_Quantity,
    ROUND(Total_Profit, 2) AS Profit,
    Performance_Tier
FROM product_performance
ORDER BY Total_Sales ASC
LIMIT 5;
"""

run_sql(query_3)

,Product Name,Category,Revenue,Total_Quantity,Profit,Performance_Tier
0,Eureka Disposable Bags for Sanitaire Vibra Gro...,Office Supplies,1.62,2,-4.47,Low Performer
1,Avery 5,Office Supplies,5.76,2,2.82,Low Performer
2,Xerox 20,Office Supplies,6.48,1,3.11,Low Performer
3,Grip Seal Envelopes,Office Supplies,7.07,2,2.39,Low Performer
4,Avery Hi-Liter Pen Style Six-Color Fluorescent...,Office Supplies,7.70,2,3.16,Low Performer


In [12]:
query_4 = """
SELECT 
    Category,
    COUNT(DISTINCT "Product Name") AS Product_Count,
    ROUND(SUM(Total_Sales), 2) AS Category_Revenue,
    ROUND(SUM(Total_Profit), 2) AS Category_Profit
FROM product_performance
GROUP BY Category
ORDER BY Category_Revenue DESC;
"""

run_sql(query_4)

,Category,Product_Count,Category_Revenue,Category_Profit
0,Technology,412,836154.03,145454.95
1,Furniture,380,741999.80,18451.27
2,Office Supplies,1058,719047.03,122490.80


In [13]:
conn.close()
print("Database connection closed. Day 2 SQL Analysis Complete!")

Database connection closed. Day 2 SQL Analysis Complete!
